# ME-TST+ 六方法融合筛选
请先完成原 ME-TST+ notebook 的官方仓库、依赖和路径设置。这里只读取已有官方响应，不训练模型。上传 metst_fusion_screening.zip 后按顺序运行。both 仅指 ME-TST+ 的 SAMMLV 和 CAS(ME)3。

In [ ]:
from google.colab import drive, files
from pathlib import Path
import zipfile, io, subprocess, sys
drive.mount('/content/drive')
uploaded = files.upload()
name = 'metst_fusion_screening.zip'
assert name in uploaded, '请上传 metst_fusion_screening.zip'
task_dir = Path('/content/glsd_fusion_screening')
task_dir.mkdir(exist_ok=True)
allowed = {'run_fusion_screening.py', 'official_response_component_ablation.py', 'README.md', 'fusion_screening_colab.ipynb'}
with zipfile.ZipFile(io.BytesIO(uploaded[name])) as archive:
    assert set(archive.namelist()) == allowed, '脚本包内容与预期不一致'
    archive.extractall(task_dir)
print('脚本已准备:', task_dir)


In [ ]:
subprocess.run([sys.executable, str(task_dir / 'run_fusion_screening.py'),
                '--setting', 'both', '--check-inputs'], check=True)


In [ ]:
from datetime import datetime, timezone
import os
run_tag = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
output_base = Path('/content/drive/MyDrive/GLSD_FUSION_SCREENING')
output_base.mkdir(parents=True, exist_ok=True)
screening_output = output_base / ('screen_' + run_tag)
screening_log = output_base / ('screen_' + run_tag + '.log')
command = [sys.executable, '-u', str(task_dir / 'run_fusion_screening.py'),
           '--setting', 'both', '--output', str(screening_output)]
env = dict(os.environ)
env['PYTHONPATH'] = os.pathsep.join(dict.fromkeys([os.getcwd()] + [p for p in sys.path if p] + [env.get('PYTHONPATH', '')]))
print('输出目录:', screening_output)
print('完整日志:', screening_log)
with screening_log.open('x', encoding='utf-8') as log:
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env)
    try:
        for line in process.stdout:
            print(line, end='')
            log.write(line)
            log.flush()
        returncode = process.wait()
    except BaseException:
        process.terminate()
        raise
assert returncode == 0, f'运行停止，请查看日志: {screening_log}'
assert (screening_output / 'completion.json').is_file(), '缺少完整完成标志'


In [ ]:
import pandas as pd, json
display(pd.read_csv(screening_output / 'screening_summary_full.csv'))
display(pd.read_csv(screening_output / 'paired_comparisons.csv'))
decision = json.loads((screening_output / 'decision.json').read_text())
print('筛选建议:', decision['recommendation'])
print('标签是预算建议，不能代替对 TP/FP/FN、置信区间和两数据集差异的判断。')
